In [2]:
# Install missing dependencies for Jupyter kernel
%pip install geopandas matplotlib pandas


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


03_plot_membership_certainty_map.py

Tugas Orang 5 — Peta kepastian/ambiguitas membership FCM per provinsi.

Input (data real dari Orang 1 & 4):
- data/external/indonesia_38_provinces.geojson
- data/interim/fcm_membership_geojson.csv   ← output script 01
    Kolom kunci: province_name_geojson, crisp_cluster, maximum_membership,
                 membership_margin, membership_status

Logika visual:
- Warna dasar = crisp_cluster (biru / merah)
- Opasitas (alpha) dikodekan dari maximum_membership:
    alpha = 0.30 + 0.70 × maximum_membership
    → semakin yakin = semakin solid; semakin ambigu = semakin pudar
- Status keanggotaan (dari Orang 4):
    "Ambigu tinggi"   → outline hitam tebal putus-putus, diberi label nama
    "Transisi moderat"→ outline abu gelap tipis putus-putus
    "Keanggotaan kuat"→ outline putih normal

Output:
- outputs/figures/peta_kepastian_membership.png

In [3]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

PROJECT_ROOT    = Path().resolve().parent
GEOJSON_PATH    = PROJECT_ROOT / "data" / "external" / "indonesia_38_provinces.geojson"
HARMONIZED_PATH = PROJECT_ROOT / "data" / "interim" / "fcm_membership_geojson.csv"
OUTPUT_PATH     = PROJECT_ROOT / "outputs" / "figures" / "peta_kepastian_membership.png"

CLUSTER_BASE_COLORS = {1: "#2563EB", 2: "#DC2626"}
NO_DATA_COLOR = "#CCCCCC"

STATUS_STYLE = {
    "Ambigu tinggi":    dict(edgecolor="#111111", linewidth=2.0, linestyle="--"),
    "Transisi moderat": dict(edgecolor="#555555", linewidth=1.2, linestyle="--"),
    "Keanggotaan kuat": dict(edgecolor="#FFFFFF",  linewidth=0.4, linestyle="-"),
}

In [4]:
def make_rgba(hex_color: str, alpha: float) -> tuple:
    """Konversi hex + alpha ke tuple RGBA float."""
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4))
    return (r, g, b, float(np.clip(alpha, 0.0, 1.0)))

In [5]:
def main():
    gdf = gpd.read_file(GEOJSON_PATH)
    df  = pd.read_csv(HARMONIZED_PATH)

    merged = gdf.merge(
        df[["province_name_geojson", "crisp_cluster", "maximum_membership", "membership_status"]],
        left_on="PROVINSI",
        right_on="province_name_geojson",
        how="left",
    )

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    fig.patch.set_facecolor("white")

    for _, row in merged.iterrows():
        geom = gpd.GeoSeries([row.geometry], crs=gdf.crs)

        if pd.isna(row["crisp_cluster"]):
            geom.plot(ax=ax, color=NO_DATA_COLOR, edgecolor="white", linewidth=0.4)
            continue

        base_hex = CLUSTER_BASE_COLORS[int(row["crisp_cluster"])]
        alpha    = 0.30 + 0.70 * float(row["maximum_membership"])
        rgba     = make_rgba(base_hex, alpha)

        status   = str(row["membership_status"]) if not pd.isna(row["membership_status"]) else "Keanggotaan kuat"
        style    = STATUS_STYLE.get(status, STATUS_STYLE["Keanggotaan kuat"])

        geom.plot(ax=ax, color=rgba, **style)

    # ── Legenda ───────────────────────────────────────────────────────────────
    cluster_handles = [
        mpatches.Patch(color=c, label=lbl)
        for c, lbl in [
            ("#2563EB", "Klaster 1 — Profil risiko faktor lebih rendah"),
            ("#DC2626", "Klaster 2 — Profil risiko faktor lebih tinggi"),
        ]
    ]
    status_handles = [
        Line2D([0], [0], color="#111111", linewidth=2.0, linestyle="--",
               label="Ambigu tinggi (margin < 0.05)"),
        Line2D([0], [0], color="#555555", linewidth=1.2, linestyle="--",
               label="Transisi moderat (margin 0.05–0.45)"),
        Line2D([0], [0], color="#AAAAAA", linewidth=0.4, linestyle="-",
               label="Keanggotaan kuat (margin ≥ 0.45)"),
        mpatches.Patch(color=NO_DATA_COLOR, label="Tidak ada data FCM"),
    ]
    opacity_handles = [
        mpatches.Patch(color="#2563EB", alpha=1.0, label="Opacity solid = membership tinggi"),
        mpatches.Patch(color="#2563EB", alpha=0.45, label="Opacity pudar = membership rendah"),
    ]

    legend1 = ax.legend(
        handles=cluster_handles + status_handles,
        loc="lower left",
        fontsize=8.5,
        frameon=True,
        framealpha=0.92,
        title="Klaster & Status Membership",
        title_fontsize=9,
    )
    ax.add_artist(legend1)
    legend2 = ax.legend(
        handles=opacity_handles,
        loc="lower right",
        fontsize=8.5,
        frameon=True,
        framealpha=0.92,
        title="Kode Opacity",
        title_fontsize=9,
    )

    ax.set_title(
        "Peta Kepastian Membership FCM per Provinsi\n"
        "Opacity ∝ nilai max-membership · garis putus-putus = provinsi ambigu",
        fontsize=13,
        weight="bold",
        pad=14,
    )
    ax.axis("off")
    fig.tight_layout(pad=1.5)

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_PATH, dpi=220, bbox_inches="tight", facecolor="white")
    print(f"Peta kepastian membership disimpan → {OUTPUT_PATH}")
    plt.close(fig)

### Jalankan Pipeline

In [6]:
if __name__ == "__main__":
    main()

Peta kepastian membership disimpan → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\figures\peta_kepastian_membership.png
